# Module 03 — State, Memory & Recovery (Colab)

NovaBridge's agent writes each client's **account summary**. Two clients are processed at once, and the agent keeps its working state in a shared **memory** object. Watch Tenant Alpha's summary end up showing **Tenant Beta's balance**.

> ▶️ **Run every cell in order, top to bottom.** If a cell errors, re-run the setup cell first (it is idempotent), then continue.

## 1. Set up (Postgres + recorded model outputs, ~2 min)

In [ ]:
%cd /content
!rm -rf repo
!git clone https://github.com/sanbhaumik/workshop-designing-data-infra-for-ai-agents.git repo
%cd repo

In [ ]:
!SKIP_OLLAMA=1 bash setup.sh

In [ ]:
import os
os.environ['NOVA_LLM'] = 'frozen'
os.environ['DATABASE_URL'] = 'postgresql://postgres@localhost:5432/nova'
!python preflight.py

## 2. Two tenants, one shared memory

Two clients with different balances. The agent keeps working state in a `memory` object keyed by `run_id`.

In [ ]:
import sys, importlib
sys.path.insert(0, 'modules/03_state')

from nova.llm import get_llm
from nova.store import get_store
from nova.agent import summary_prompt, load_document, format_summary, SharedState
import your_fix

llm = get_llm()
store = get_store(); store.init_schema(); store.reset_demo()

doc_a = load_document('alpha', 'account_note.md')
doc_b = load_document('beta', 'account_note.md')
print(doc_a)
print(doc_b)

The naive `memory` (`SharedState`) **ignores the run_id** — every run gets the same dict. Watch what that does when Alpha's run and Beta's run interleave:

In [ ]:
memory = SharedState()

# Alpha's run: read, then reason (write its summary into memory)
memory.set('run-a', {'client_id': 'alpha'})
memory.set('run-a', {'client_id': 'alpha', 'last_response': llm.complete(summary_prompt('alpha', doc_a))})
print("Alpha reasoned. memory for run-a:", memory.get('run-a')['client_id'])

# Beta's run interleaves: read, then reason
memory.set('run-b', {'client_id': 'beta'})
memory.set('run-b', {'client_id': 'beta', 'last_response': llm.complete(summary_prompt('beta', doc_b))})

# back to Alpha's run — what does ITS memory say now?
print("Beta reasoned.  memory for run-a:", memory.get('run-a')['client_id'], '  <-- it flipped to beta!')

Alpha's run resumes and **saves its summary** — from memory that now holds Beta's data:

In [ ]:
m = memory.get('run-a')
alpha_summary = format_summary(m['client_id'], m['last_response'])
store.set_summary('alpha', alpha_summary)
print(alpha_summary)

See the leaked row in the **real database** — Alpha's summary, holding Beta's $1.1M:

In [ ]:
!psql "$DATABASE_URL" -c "SELECT client_id, content FROM summaries;"

### The aha

`SharedState` returned the **same dict** for `run-a` and `run-b`, so Beta's run overwrote Alpha's working memory. The leak happened **inside the agent's memory**, before any per-tenant check ever ran. This is the failure that bites real multi-tenant agent systems.

The fix: give each run its own memory.

## 3. Fix it

Here's the code you'll fix. `IsolatedState` currently behaves exactly like `SharedState`.

In [ ]:
print(open('modules/03_state/your_fix.py').read())

Edit the cell below so `get`/`set` give each `run_id` its **own** dict, then run it to save:

In [ ]:
%%writefile modules/03_state/your_fix.py
class IsolatedState:
    def __init__(self):
        self._data = {}

    def get(self, run_id):
        # TODO: give each run_id its OWN dict instead of one shared dict
        return self._data

    def set(self, run_id, data):
        # TODO: store this run_id's data on its own, not merged into a shared dict
        self._data.update(data)

Reload your fix and run the **exact same interleaving** — this time watch Alpha's memory stay Alpha:

In [ ]:
importlib.reload(your_fix)
memory = your_fix.IsolatedState()

memory.set('run-a', {'client_id': 'alpha', 'last_response': llm.complete(summary_prompt('alpha', doc_a))})
memory.set('run-b', {'client_id': 'beta',  'last_response': llm.complete(summary_prompt('beta', doc_b))})

print("memory for run-a:", memory.get('run-a').get('client_id'), ' (should be alpha)')
m = memory.get('run-a')
print(format_summary(m['client_id'], m['last_response']))

Alpha's memory stayed Alpha, so Alpha's summary is clean. **That** is memory isolation.

## 4. Prove it

In [ ]:
!python -m pytest modules/03_state/test_state.py -v

### Optional: run the real local model

Everything above replayed the model's answers from recordings. To generate them live with the real open-source model, run `!bash setup.sh`, set `os.environ['NOVA_LLM']='ollama'`, and re-run from section 2.